# A Haskell primer for C# developers

## Up the cliff face
The previous rung (F#) was familiar enough. Same runtime as C#, same
NuGet, same `dotnet test`. Discriminated unions instead of class
hierarchies, but you could squint and see how the C# code maps over.

Haskell is the cliff face above that. Different runtime, different
build tool, no `null`, no `class`, no `void`. The thing that's worth
the climb: **Haskell is the language where the type system was the
design goal**, not a feature bolted onto something else. Pure
functions are the default, immutability is the default, side effects
are tracked in the function's *signature* — so the compiler knows
which functions can do I/O and which can't.

Our fuel engine maps almost 1:1 onto Haskell's natural shape. A row
decision is one of six things — that's a sum type. A non-empty list
of warnings is a `NonEmpty`. A classifier that does no I/O is a
function whose signature simply has no `IO` in it. The work isn't
translating the design into the language; the work is reading the
language at all.

So this chapter is the syntax tour. Just enough to make the next
chapter — where we look at what Haskell makes **unrepresentable** —
legible.

## The five-minute syntax tour

### `data` — sum types as a first-class declaration

A C# enum has names but no payload. A C# sealed-record hierarchy has
payload but takes 50 lines. Haskell's `data` does both, on one
declaration:

In [ ]:
data RowDecision
  = Accepted FuelTransaction
  | AcceptedWithWarnings FuelTransaction (NonEmpty ValidationWarning)
  | Quarantined FuelTransaction (NonEmpty QuarantineReason)
  | SkippedDuplicate SkippedDuplicate
  | Rejected RejectedRow
  | Fatal FatalError
  deriving stock (Eq, Show)

Read it like English: "a `RowDecision` *is* one of: an `Accepted`
carrying a `FuelTransaction`, **or** an `AcceptedWithWarnings` carrying
a transaction plus a non-empty list of warnings, **or** …" The `|` is
"or." Each branch is called a **constructor**.

F# (chapter 5) is nine lines for the same thing. Haskell is eight.
Idiomatic C# (chapter 4) is roughly fifty, because each case needs its
own nested `sealed record` and the no-empty-list invariant becomes a
runtime check in a constructor. The line count is a proxy for **what
the language has built in vs. what you have to assemble**. Haskell has
sum types built in.

### `deriving stock (Eq, Show)` — equality and printing for free

C# `record` gives you structural equality and a `ToString()` for free.
Haskell did it first, more generally. `deriving stock (Eq, Show)`
says "compiler, please write the obvious structural-equality and
print-to-string implementations for me." The `stock` keyword (enabled
by the `DerivingStrategies` extension this project uses) is
hygiene — explicit about *which* derivation strategy to apply.

You'll see this on every type in the codebase. It's not noise; it's
the deal you signed up for.

### Type signatures — `::` and arrows

C# writes `Foo MethodName(A a, B b, C c)`. Haskell writes the type on
its own line, with arrows:

In [ ]:
classifyRow :: ValidationConfig -> UploadMode -> RowContext -> RowDecision
classifyRow config mode context =
  case (...) of
    ...

Read the `::` as "has type" and the `->` as "function returning":

> `classifyRow` has type — given a `ValidationConfig`, then an
> `UploadMode`, then a `RowContext`, produces a `RowDecision`.

The type signature is on the line above; the function body is below.
The reason the arrows chain the way they do (rather than using commas)
is **currying** — every multi-argument Haskell function is, under the
hood, a function that takes the first argument and returns a function
of the rest. That's why you can write `fmap (classifyRow config mode)
contexts` later in `DecisionEngine.hs` — `classifyRow config mode` is
itself a valid value, a function of type `RowContext -> RowDecision`,
ready to be mapped over a list. C# can do the same with explicit
lambdas; Haskell makes it the default shape.

### Pattern matching with `case ... of`

C# 9 gave you `switch` expressions. Haskell has had this since 1990.
The classifier opens like this:

In [ ]:
classifyRow config mode context =
  case (contextVehicleLookup context, contextDuplicateCheck context) of
    (VehicleLookupFatal fatalError, _) ->
      Fatal fatalError
    (_, DuplicateCheckFatal fatalError) ->
      Fatal fatalError
    _ ->
      classifyValidated

Three things to notice:

1. **It's an expression.** The whole `case … of …` block *is* a
   value. You can return it, assign it, pass it to another function.
2. **Wildcards.** `_` matches anything without binding.
3. **Tuples.** `(a, b)` builds a pair; `(a, b) ->` destructures one.

The compiler refuses to ship if you've left a constructor unmatched —
provided the project has `-Wincomplete-uni-patterns` on, which this
one does. Footgun 7 (the `switch` statement with no default) cannot
exist in Haskell.

### `Maybe a` — explicit optionality

Haskell has no `null`. There is no value of type `Vehicle` that
represents "no vehicle." If you want optionality, you ask for it in
the type:

In [ ]:
data Maybe a = Nothing | Just a

`Maybe Vehicle` is a sum type with two constructors: `Nothing` (it's
absent) or `Just vehicle` (it's present, here it is). You can't call
a `Vehicle` method on a `Maybe Vehicle` — the compiler refuses. You
have to pattern-match it open first:

In [ ]:
case maybeVehicle of
  Nothing -> ...
  Just vehicle -> ... -- now we have the vehicle

This is the same idea as F#'s `Option<'T>` from chapter 5, except
**there is no opt-out**. C# 8 nullable reference types are opt-in
and the compiler treats violations as warnings. Haskell's `Maybe` is
the only way to express "might not be there," and the compiler treats
violations as type errors.

### `Either e a` — typed errors instead of exceptions

The other workhorse. `Either e a` is either a `Left e` (failure) or a
`Right a` (success). The boundary parser uses it everywhere. C# would
throw a `FormatException`. F# would return `Result<UploadMode,
ParseError>`. Haskell returns `Either [FuelUploadMappingError]
UploadMode`. The "error path" is a value, not an exception, and the
type system forces every caller to consider it.

### `NonEmpty a` — a list with at least one element, at the type level

The single most useful generic type in this codebase. Defined in the
standard library as:

In [ ]:
data NonEmpty a = a :| [a]

A non-empty list of `a` is one `a` followed by a (possibly empty)
regular list of `a`. The `:|` is the constructor. There is **no value
of type `NonEmpty a` that is empty**. The compiler will not let you
build one.

That's why `RowDecision` can say:

In [ ]:
| Quarantined FuelTransaction (NonEmpty QuarantineReason)

and the empty-quarantine-reasons case literally cannot exist. In C# we
had to write a constructor that throws `ArgumentException` when the
list is empty — a runtime guard. In Haskell the type itself is the
guard. Chapter 8 returns to this.

### `IO ()` — side effects in the signature

Pure functions in Haskell have signatures like `A -> B`. Functions
that do I/O — read a file, log to console, hit the network — have
signatures with `IO` in them:

In [ ]:
main :: IO ()
putStrLn :: String -> IO ()
readFile :: FilePath -> IO String

`IO String` means "an action that, when run, produces a `String`."
The crucial property: **`IO` is contagious**. If you call an `IO`
function, your function is `IO` too. There is no way to call
`putStrLn` from inside a function declared `:: A -> B` without
changing the signature to `:: A -> IO B`.

Our entire decision engine is pure:

In [ ]:
classifyRow :: ValidationConfig -> UploadMode -> RowContext -> RowDecision

No `IO` anywhere. The compiler guarantees no row classifier in this
codebase can ever log, mutate, or call a database. Footgun 1 (the
rogue logger touching a null `Vehicle`) is structurally impossible
because *there is no way for a logger to live inside `classifyRow`*.

### Record syntax and field-accessors-as-functions

In [ ]:
data PreviousAttempt = PreviousAttempt
  { previousTransactionId :: TransactionId
  , previousCanonicalizationState :: CanonicalizationState
  , previousFinalizationState :: FinalizationState
  }
  deriving stock (Eq, Show)

Familiar shape — three fields with types. The Haskell-specific part:
**every field name is automatically a top-level function**. The
declaration above generates:

In [ ]:
previousFinalizationState :: PreviousAttempt -> FinalizationState

So instead of writing `previousAttempt.PreviousFinalizationState`,
you write `previousFinalizationState previousAttempt` — the field
*is* the accessor function. You'll see this all over
`DuplicatePolicy.hs`:

In [ ]:
| previousFinalizationState previousAttempt == FailedRetryable =
    UploadDuplicate

Read: "if the result of calling `previousFinalizationState` on
`previousAttempt` equals `FailedRetryable`, then `UploadDuplicate`."

The catch: field names are global within their module, so this
codebase prefixes everything (`previousTransactionId`,
`parsedRowNumber`, `transactionAmount`). That prefix is the workaround
for the global-name rule.

### `newtype` — zero-cost wrappers for primitives

In [ ]:
newtype RowNumber = RowNumber Int
  deriving stock (Eq, Ord, Show)

A `newtype` is like `data` restricted to exactly one constructor with
exactly one field. The compiler erases the wrapper at runtime — a
`RowNumber` is just an `Int` in memory — but to the type system,
`RowNumber` and `Int` are incompatible. Zero overhead, full safety.
Same trick as the idiomatic C# `record struct RowNumber(int Value)`,
shorter syntax and free `Eq`/`Ord`/`Show`.

## The module layout

<div class="mermaid-diagram" style="width:100%">
<svg id="mermaid-figure-1" xmlns="http://www.w3.org/2000/svg" xlink="http://www.w3.org/1999/xlink" class="flowchart" viewbox="0 0 1306.359375 765" role="graphics-document document" aria-roledescription="flowchart-v2" style="width:100%;height:auto;max-width:100%;display:block">
<style>#mermaid-figure-1{font-family:"trebuchet ms",verdana,arial,sans-serif;font-size:16px;fill:#000000;}@keyframes edge-animation-frame{from{stroke-dashoffset:0;}}@keyframes dash{to{stroke-dashoffset:0;}}#mermaid-figure-1 .edge-animation-slow{stroke-dasharray:9,5!important;stroke-dashoffset:900;animation:dash 50s linear infinite;stroke-linecap:round;}#mermaid-figure-1 .edge-animation-fast{stroke-dasharray:9,5!important;stroke-dashoffset:900;animation:dash 20s linear infinite;stroke-linecap:round;}#mermaid-figure-1 .error-icon{fill:#552222;}#mermaid-figure-1 .error-text{fill:#552222;stroke:#552222;}#mermaid-figure-1 .edge-thickness-normal{stroke-width:1px;}#mermaid-figure-1 .edge-thickness-thick{stroke-width:3.5px;}#mermaid-figure-1 .edge-pattern-solid{stroke-dasharray:0;}#mermaid-figure-1 .edge-thickness-invisible{stroke-width:0;fill:none;}#mermaid-figure-1 .edge-pattern-dashed{stroke-dasharray:3;}#mermaid-figure-1 .edge-pattern-dotted{stroke-dasharray:2;}#mermaid-figure-1 .marker{fill:#666;stroke:#666;}#mermaid-figure-1 .marker.cross{stroke:#666;}#mermaid-figure-1 svg{font-family:"trebuchet ms",verdana,arial,sans-serif;font-size:16px;}#mermaid-figure-1 p{margin:0;}#mermaid-figure-1 .label{font-family:"trebuchet ms",verdana,arial,sans-serif;color:#000000;}#mermaid-figure-1 .cluster-label text{fill:#333;}#mermaid-figure-1 .cluster-label span{color:#333;}#mermaid-figure-1 .cluster-label span p{background-color:transparent;}#mermaid-figure-1 .label text,#mermaid-figure-1 span{fill:#000000;color:#000000;}#mermaid-figure-1 .node rect,#mermaid-figure-1 .node circle,#mermaid-figure-1 .node ellipse,#mermaid-figure-1 .node polygon,#mermaid-figure-1 .node path{fill:#eee;stroke:#999;stroke-width:1px;}#mermaid-figure-1 .rough-node .label text,#mermaid-figure-1 .node .label text,#mermaid-figure-1 .image-shape .label,#mermaid-figure-1 .icon-shape .label{text-anchor:middle;}#mermaid-figure-1 .node .katex path{fill:#000;stroke:#000;stroke-width:1px;}#mermaid-figure-1 .rough-node .label,#mermaid-figure-1 .node .label,#mermaid-figure-1 .image-shape .label,#mermaid-figure-1 .icon-shape .label{text-align:center;}#mermaid-figure-1 .node.clickable{cursor:pointer;}#mermaid-figure-1 .root .anchor path{fill:#666!important;stroke-width:0;stroke:#666;}#mermaid-figure-1 .arrowheadPath{fill:#333333;}#mermaid-figure-1 .edgePath .path{stroke:#666;stroke-width:2.0px;}#mermaid-figure-1 .flowchart-link{stroke:#666;fill:none;}#mermaid-figure-1 .edgeLabel{background-color:white;text-align:center;}#mermaid-figure-1 .edgeLabel p{background-color:white;}#mermaid-figure-1 .edgeLabel rect{opacity:0.5;background-color:white;fill:white;}#mermaid-figure-1 .labelBkg{background-color:rgba(255, 255, 255, 0.5);}#mermaid-figure-1 .cluster rect{fill:hsl(0, 0%, 98.9215686275%);stroke:#707070;stroke-width:1px;}#mermaid-figure-1 .cluster text{fill:#333;}#mermaid-figure-1 .cluster span{color:#333;}#mermaid-figure-1 div.mermaidTooltip{position:absolute;text-align:center;max-width:200px;padding:2px;font-family:"trebuchet ms",verdana,arial,sans-serif;font-size:12px;background:hsl(-160, 0%, 93.3333333333%);border:1px solid #707070;border-radius:2px;pointer-events:none;z-index:100;}#mermaid-figure-1 .flowchartTitleText{text-anchor:middle;font-size:18px;fill:#000000;}#mermaid-figure-1 rect.text{fill:none;stroke-width:0;}#mermaid-figure-1 .icon-shape,#mermaid-figure-1 .image-shape{background-color:white;text-align:center;}#mermaid-figure-1 .icon-shape p,#mermaid-figure-1 .image-shape p{background-color:white;padding:2px;}#mermaid-figure-1 .icon-shape rect,#mermaid-figure-1 .image-shape rect{opacity:0.5;background-color:white;fill:white;}#mermaid-figure-1 .label-icon{display:inline-block;height:1em;overflow:visible;vertical-align:-0.125em;}#mermaid-figure-1 .node .label-icon path{fill:currentColor;stroke:revert;stroke-width:revert;}#mermaid-figure-1 :root{--mermaid-font-family:"trebuchet ms",verdana,arial,sans-serif;}</style>
<g><marker id="mermaid-1779381580185_flowchart-v2-pointEnd" class="marker flowchart-v2" viewbox="0 0 10 10" refx="5" refy="5" markerunits="userSpaceOnUse" markerwidth="8" markerheight="8" orient="auto"><path d="M 0 0 L 10 5 L 0 10 z" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></path></marker><marker id="mermaid-1779381580185_flowchart-v2-pointStart" class="marker flowchart-v2" viewbox="0 0 10 10" refx="4.5" refy="5" markerunits="userSpaceOnUse" markerwidth="8" markerheight="8" orient="auto"><path d="M 0 5 L 10 10 L 10 0 z" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></path></marker><marker id="mermaid-1779381580185_flowchart-v2-circleEnd" class="marker flowchart-v2" viewbox="0 0 10 10" refx="11" refy="5" markerunits="userSpaceOnUse" markerwidth="11" markerheight="11" orient="auto"><circle cx="5" cy="5" r="5" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></circle></marker><marker id="mermaid-1779381580185_flowchart-v2-circleStart" class="marker flowchart-v2" viewbox="0 0 10 10" refx="-1" refy="5" markerunits="userSpaceOnUse" markerwidth="11" markerheight="11" orient="auto"><circle cx="5" cy="5" r="5" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></circle></marker><marker id="mermaid-1779381580185_flowchart-v2-crossEnd" class="marker cross flowchart-v2" viewbox="0 0 11 11" refx="12" refy="5.2" markerunits="userSpaceOnUse" markerwidth="11" markerheight="11" orient="auto"><path d="M 1,1 l 9,9 M 10,1 l -9,9" class="arrowMarkerPath" style="stroke-width: 2; stroke-dasharray: 1, 0;"></path></marker><marker id="mermaid-1779381580185_flowchart-v2-crossStart" class="marker cross flowchart-v2" viewbox="0 0 11 11" refx="-1" refy="5.2" markerunits="userSpaceOnUse" markerwidth="11" markerheight="11" orient="auto"><path d="M 1,1 l 9,9 M 10,1 l -9,9" class="arrowMarkerPath" style="stroke-width: 2; stroke-dasharray: 1, 0;"></path></marker><g class="root"><g class="clusters"><g class="cluster" id="Reports" data-look="classic"><rect style="" x="931.171875" y="291" width="367.1875" height="104"></rect><g class="cluster-label" transform="translate(1057.8515625, 291)"><foreignobject width="113.828125" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Projections (V3)
</p>
</span>
</div>
</foreignobject></g></g><g class="cluster" id="Domain" data-look="classic"><rect style="" x="8" y="445" width="1271.875" height="312"></rect><g class="cluster-label" transform="translate(571.4609375, 445)"><foreignobject width="144.953125" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Domain (pure types)
</p>
</span>
</div>
</foreignobject></g></g><g class="cluster" id="Engine" data-look="classic"><rect style="" x="155.96484375" y="162" width="755.20703125" height="233"></rect><g class="cluster-label" transform="translate(485.091796875, 162)"><foreignobject width="96.953125" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Engine (pure)
</p>
</span>
</div>
</foreignobject></g></g><g class="cluster" id="Boundary" data-look="classic"><rect style="" x="72.34765625" y="8" width="382.58984375" height="104"></rect><g class="cluster-label" transform="translate(163.642578125, 8)"><foreignobject width="200" height="48">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table; white-space: break-spaces; line-height: 1.5; max-width: 200px; text-align: center; width: 200px;">
<span class="nodeLabel">
<p>
FuelUpload.Api — DTO &lt;-&gt; domain
</p>
</span>
</div>
</foreignobject></g></g></g><g class="edgePaths"><path d="M355.702,87L363.305,91.167C370.908,95.333,386.114,103.667,393.717,112C401.32,120.333,401.32,128.667,401.32,137C401.32,145.333,401.32,153.667,401.32,161.333C401.32,169,401.32,176,401.32,179.5L401.32,183" id="L_Api_DE_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_Api_DE_0" data-points="W3sieCI6MzU1LjcwMTY5NzcxNjM0NjEzLCJ5Ijo4N30seyJ4Ijo0MDEuMzIwMzEyNSwieSI6MTEyfSx7IngiOjQwMS4zMjAzMTI1LCJ5IjoxMzd9LHsieCI6NDAxLjMyMDMxMjUsInkiOjE2Mn0seyJ4Ijo0MDEuMzIwMzEyNSwieSI6MTg3fV0=" marker-end="url(#mermaid-1779381580185_flowchart-v2-pointEnd)"></path><path d="M254.199,75.934L234.493,81.945C214.788,87.956,175.376,99.978,155.671,110.156C135.965,120.333,135.965,128.667,135.965,137C135.965,145.333,135.965,153.667,135.965,166.5C135.965,179.333,135.965,196.667,135.965,214C135.965,231.333,135.965,248.667,135.965,261.5C135.965,274.333,135.965,282.667,135.965,295.5C135.965,308.333,135.965,325.667,135.965,343C135.965,360.333,135.965,377.667,135.965,390.5C135.965,403.333,135.965,411.667,135.965,420C135.965,428.333,135.965,436.667,196.124,447.703C256.283,458.739,376.602,472.477,436.761,479.347L496.92,486.216" id="L_Api_DEC_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_Api_DEC_0" data-points="W3sieCI6MjU0LjE5OTIxODc1LCJ5Ijo3NS45MzM2Mzg4NjM0MjgwNH0seyJ4IjoxMzUuOTY0ODQzNzUsInkiOjExMn0seyJ4IjoxMzUuOTY0ODQzNzUsInkiOjEzN30seyJ4IjoxMzUuOTY0ODQzNzUsInkiOjE2Mn0seyJ4IjoxMzUuOTY0ODQzNzUsInkiOjIxNH0seyJ4IjoxMzUuOTY0ODQzNzUsInkiOjI2Nn0seyJ4IjoxMzUuOTY0ODQzNzUsInkiOjI5MX0seyJ4IjoxMzUuOTY0ODQzNzUsInkiOjM0M30seyJ4IjoxMzUuOTY0ODQzNzUsInkiOjM5NX0seyJ4IjoxMzUuOTY0ODQzNzUsInkiOjQyMH0seyJ4IjoxMzUuOTY0ODQzNzUsInkiOjQ0NX0seyJ4Ijo1MDAuODk0NTMxMjUsInkiOjQ4Ni42Njk3NjAzNDAzNjEzfV0=" marker-end="url(#mermaid-1779381580185_flowchart-v2-pointEnd)"></path><path d="M385.77,241L383.37,245.167C380.97,249.333,376.171,257.667,373.771,266C371.371,274.333,371.371,282.667,371.371,290.333C371.371,298,371.371,305,371.371,308.5L371.371,312" id="L_DE_DP_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_DE_DP_0" data-points="W3sieCI6Mzg1Ljc2OTc1NjYxMDU3NjksInkiOjI0MX0seyJ4IjozNzEuMzcxMDkzNzUsInkiOjI2Nn0seyJ4IjozNzEuMzcxMDkzNzUsInkiOjI5MX0seyJ4IjozNzEuMzcxMDkzNzUsInkiOjMxNn1d" marker-end="url(#mermaid-1779381580185_flowchart-v2-pointEnd)"></path><path d="M497.586,240.34L513.215,244.617C528.845,248.894,560.104,257.447,575.734,265.89C591.363,274.333,591.363,282.667,591.363,290.333C591.363,298,591.363,305,591.363,308.5L591.363,312" id="L_DE_VAL_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_DE_VAL_0" data-points="W3sieCI6NDk3LjU4NTkzNzUsInkiOjI0MC4zNDA0MjQ2NTcyNTI2N30seyJ4Ijo1OTEuMzYzMjgxMjUsInkiOjI2Nn0seyJ4Ijo1OTEuMzYzMjgxMjUsInkiOjI5MX0seyJ4Ijo1OTEuMzYzMjgxMjUsInkiOjMxNn1d" marker-end="url(#mermaid-1779381580185_flowchart-v2-pointEnd)"></path><path d="M497.586,226.85L546.469,233.375C595.353,239.9,693.12,252.95,742.003,263.642C790.887,274.333,790.887,282.667,790.887,290.333C790.887,298,790.887,305,790.887,308.5L790.887,312" id="L_DE_SUM_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_DE_SUM_0" data-points="W3sieCI6NDk3LjU4NTkzNzUsInkiOjIyNi44NDk3MDI2OTQzMDE1Nn0seyJ4Ijo3OTAuODg2NzE4NzUsInkiOjI2Nn0seyJ4Ijo3OTAuODg2NzE4NzUsInkiOjI5MX0seyJ4Ijo3OTAuODg2NzE4NzUsInkiOjMxNn1d" marker-end="url(#mermaid-1779381580185_flowchart-v2-pointEnd)"></path><path d="M318.387,241L305.589,245.167C292.791,249.333,267.194,257.667,254.396,266C241.598,274.333,241.598,282.667,241.598,295.5C241.598,308.333,241.598,325.667,241.598,343C241.598,360.333,241.598,377.667,241.598,390.5C241.598,403.333,241.598,411.667,241.598,420C241.598,428.333,241.598,436.667,284.154,447.16C326.711,457.654,411.825,470.308,454.381,476.635L496.938,482.962" id="L_DE_DEC_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_DE_DEC_0" data-points="W3sieCI6MzE4LjM4NzM5NDgzMTczMDgsInkiOjI0MX0seyJ4IjoyNDEuNTk3NjU2MjUsInkiOjI2Nn0seyJ4IjoyNDEuNTk3NjU2MjUsInkiOjI5MX0seyJ4IjoyNDEuNTk3NjU2MjUsInkiOjM0M30seyJ4IjoyNDEuNTk3NjU2MjUsInkiOjM5NX0seyJ4IjoyNDEuNTk3NjU2MjUsInkiOjQyMH0seyJ4IjoyNDEuNTk3NjU2MjUsInkiOjQ0NX0seyJ4Ijo1MDAuODk0NTMxMjUsInkiOjQ4My41NDk5MjE4MjI2NDkxfV0=" marker-end="url(#mermaid-1779381580185_flowchart-v2-pointEnd)"></path><path d="M371.371,370L371.371,374.167C371.371,378.333,371.371,386.667,371.371,395C371.371,403.333,371.371,411.667,371.371,420C371.371,428.333,371.371,436.667,392.31,445.783C413.248,454.899,455.125,464.797,476.063,469.746L497.002,474.696" id="L_DP_DEC_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_DP_DEC_0" data-points="W3sieCI6MzcxLjM3MTA5Mzc1LCJ5IjozNzB9LHsieCI6MzcxLjM3MTA5Mzc1LCJ5IjozOTV9LHsieCI6MzcxLjM3MTA5Mzc1LCJ5Ijo0MjB9LHsieCI6MzcxLjM3MTA5Mzc1LCJ5Ijo0NDV9LHsieCI6NTAwLjg5NDUzMTI1LCJ5Ijo0NzUuNjE1NzE3ODg3NzA5MX1d" marker-end="url(#mermaid-1779381580185_flowchart-v2-pointEnd)"></path><path d="M591.363,370L591.363,374.167C591.363,378.333,591.363,386.667,591.363,395C591.363,403.333,591.363,411.667,591.363,420C591.363,428.333,591.363,436.667,591.363,444.333C591.363,452,591.363,459,591.363,462.5L591.363,466" id="L_VAL_DEC_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_VAL_DEC_0" data-points="W3sieCI6NTkxLjM2MzI4MTI1LCJ5IjozNzB9LHsieCI6NTkxLjM2MzI4MTI1LCJ5IjozOTV9LHsieCI6NTkxLjM2MzI4MTI1LCJ5Ijo0MjB9LHsieCI6NTkxLjM2MzI4MTI1LCJ5Ijo0NDV9LHsieCI6NTkxLjM2MzI4MTI1LCJ5Ijo0NzB9XQ==" marker-end="url(#mermaid-1779381580185_flowchart-v2-pointEnd)"></path><path d="M790.887,370L790.887,374.167C790.887,378.333,790.887,386.667,790.887,395C790.887,403.333,790.887,411.667,790.887,420C790.887,428.333,790.887,436.667,773.356,445.402C755.825,454.138,720.764,463.275,703.233,467.844L685.703,472.413" id="L_SUM_DEC_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_SUM_DEC_0" data-points="W3sieCI6NzkwLjg4NjcxODc1LCJ5IjozNzB9LHsieCI6NzkwLjg4NjcxODc1LCJ5IjozOTV9LHsieCI6NzkwLjg4NjcxODc1LCJ5Ijo0MjB9LHsieCI6NzkwLjg4NjcxODc1LCJ5Ijo0NDV9LHsieCI6NjgxLjgzMjAzMTI1LCJ5Ijo0NzMuNDIxOTQyOTEwODQyMjN9XQ==" marker-end="url(#mermaid-1779381580185_flowchart-v2-pointEnd)"></path><path d="M1025.078,370L1025.078,374.167C1025.078,378.333,1025.078,386.667,1025.078,395C1025.078,403.333,1025.078,411.667,1025.078,420C1025.078,428.333,1025.078,436.667,968.532,447.613C911.987,458.559,798.895,472.118,742.349,478.898L685.804,485.677" id="L_AUD_DEC_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_AUD_DEC_0" data-points="W3sieCI6MTAyNS4wNzgxMjUsInkiOjM3MH0seyJ4IjoxMDI1LjA3ODEyNSwieSI6Mzk1fSx7IngiOjEwMjUuMDc4MTI1LCJ5Ijo0MjB9LHsieCI6MTAyNS4wNzgxMjUsInkiOjQ0NX0seyJ4Ijo2ODEuODMyMDMxMjUsInkiOjQ4Ni4xNTMyOTk1Mjg5NjAzNn1d" marker-end="url(#mermaid-1779381580185_flowchart-v2-pointEnd)"></path><path d="M1198.672,370L1198.672,374.167C1198.672,378.333,1198.672,386.667,1198.672,395C1198.672,403.333,1198.672,411.667,1198.672,420C1198.672,428.333,1198.672,436.667,1113.196,448.152C1027.72,459.637,856.769,474.275,771.293,481.594L685.817,488.912" id="L_REP_DEC_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_REP_DEC_0" data-points="W3sieCI6MTE5OC42NzE4NzUsInkiOjM3MH0seyJ4IjoxMTk4LjY3MTg3NSwieSI6Mzk1fSx7IngiOjExOTguNjcxODc1LCJ5Ijo0MjB9LHsieCI6MTE5OC42NzE4NzUsInkiOjQ0NX0seyJ4Ijo2ODEuODMyMDMxMjUsInkiOjQ4OS4yNTM3MzIyMDcyOTI2N31d" marker-end="url(#mermaid-1779381580185_flowchart-v2-pointEnd)"></path><path d="M500.895,509.299L452.224,515.916C403.553,522.533,306.212,535.766,257.542,545.883C208.871,556,208.871,563,208.871,566.5L208.871,570" id="L_DEC_DUP_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_DEC_DUP_0" data-points="W3sieCI6NTAwLjg5NDUzMTI1LCJ5Ijo1MDkuMjk5MjcwODE4NDM5OTR9LHsieCI6MjA4Ljg3MTA5Mzc1LCJ5Ijo1NDl9LHsieCI6MjA4Ljg3MTA5Mzc1LCJ5Ijo1NzR9XQ==" marker-end="url(#mermaid-1779381580185_flowchart-v2-pointEnd)"></path><path d="M681.832,510.285L725.771,516.738C769.711,523.19,857.59,536.095,901.529,546.048C945.469,556,945.469,563,945.469,566.5L945.469,570" id="L_DEC_ROW_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_DEC_ROW_0" data-points="W3sieCI6NjgxLjgzMjAzMTI1LCJ5Ijo1MTAuMjg1MjM2Nzg3MjM5fSx7IngiOjk0NS40Njg3NSwieSI6NTQ5fSx7IngiOjk0NS40Njg3NSwieSI6NTc0fV0=" marker-end="url(#mermaid-1779381580185_flowchart-v2-pointEnd)"></path><path d="M681.832,505.319L761.002,512.599C840.172,519.879,998.512,534.44,1077.682,545.22C1156.852,556,1156.852,563,1156.852,566.5L1156.852,570" id="L_DEC_VEH_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_DEC_VEH_0" data-points="W3sieCI6NjgxLjgzMjAzMTI1LCJ5Ijo1MDUuMzE5MTM3OTEzMTY5NjR9LHsieCI6MTE1Ni44NTE1NjI1LCJ5Ijo1NDl9LHsieCI6MTE1Ni44NTE1NjI1LCJ5Ijo1NzR9XQ==" marker-end="url(#mermaid-1779381580185_flowchart-v2-pointEnd)"></path><path d="M500.895,506.213L430.867,513.344C360.84,520.475,220.785,534.738,150.758,550.535C80.73,566.333,80.73,583.667,80.73,601C80.73,618.333,80.73,635.667,143.16,651.255C205.589,666.843,330.447,680.686,392.876,687.608L455.306,694.53" id="L_DEC_PRIM_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_DEC_PRIM_0" data-points="W3sieCI6NTAwLjg5NDUzMTI1LCJ5Ijo1MDYuMjEyODMzMzQwOTgzMTR9LHsieCI6ODAuNzMwNDY4NzUsInkiOjU0OX0seyJ4Ijo4MC43MzA0Njg3NSwieSI6NjAxfSx7IngiOjgwLjczMDQ2ODc1LCJ5Ijo2NTN9LHsieCI6NDU5LjI4MTI1LCJ5Ijo2OTQuOTcwNDY2NDg5NTQzNH1d" marker-end="url(#mermaid-1779381580185_flowchart-v2-pointEnd)"></path><path d="M208.871,628L208.871,632.167C208.871,636.333,208.871,644.667,249.947,655.099C291.023,665.532,373.175,678.065,414.251,684.331L455.327,690.597" id="L_DUP_PRIM_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_DUP_PRIM_0" data-points="W3sieCI6MjA4Ljg3MTA5Mzc1LCJ5Ijo2Mjh9LHsieCI6MjA4Ljg3MTA5Mzc1LCJ5Ijo2NTN9LHsieCI6NDU5LjI4MTI1LCJ5Ijo2OTEuMjAwMTUzNTU4NzgyMX1d" marker-end="url(#mermaid-1779381580185_flowchart-v2-pointEnd)"></path><path d="M945.469,628L945.469,632.167C945.469,636.333,945.469,644.667,895.252,655.432C845.036,666.197,744.602,679.395,694.386,685.993L644.169,692.592" id="L_ROW_PRIM_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_ROW_PRIM_0" data-points="W3sieCI6OTQ1LjQ2ODc1LCJ5Ijo2Mjh9LHsieCI6OTQ1LjQ2ODc1LCJ5Ijo2NTN9LHsieCI6NjQwLjIwMzEyNSwieSI6NjkzLjExMzA4MzEzNDI2NjV9XQ==" marker-end="url(#mermaid-1779381580185_flowchart-v2-pointEnd)"></path><path d="M1156.852,628L1156.852,632.167C1156.852,636.333,1156.852,644.667,1071.408,656.152C985.964,667.637,815.076,682.274,729.632,689.592L644.189,696.911" id="L_VEH_PRIM_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_VEH_PRIM_0" data-points="W3sieCI6MTE1Ni44NTE1NjI1LCJ5Ijo2Mjh9LHsieCI6MTE1Ni44NTE1NjI1LCJ5Ijo2NTN9LHsieCI6NjQwLjIwMzEyNSwieSI6Njk3LjI1MTg1OTQ3NzU0NDd9XQ==" marker-end="url(#mermaid-1779381580185_flowchart-v2-pointEnd)"></path></g><g class="edgeLabels"><g class="edgeLabel"><g class="label" data-id="L_Api_DE_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_Api_DEC_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_DE_DP_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_DE_VAL_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_DE_SUM_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_DE_DEC_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_DP_DEC_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_VAL_DEC_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_SUM_DEC_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_AUD_DEC_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_REP_DEC_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_DEC_DUP_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_DEC_ROW_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_DEC_VEH_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_DEC_PRIM_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_DUP_PRIM_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_ROW_PRIM_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_VEH_PRIM_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g></g><g class="nodes"><g class="node default" id="flowchart-Api-0" transform="translate(306.43359375, 60)"><rect class="basic label-container" style="" x="-52.234375" y="-27" width="104.46875" height="54"></rect><g class="label" style="" transform="translate(-22.234375, -12)"><rect></rect><foreignobject width="44.46875" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Api.hs
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-DE-1" transform="translate(401.3203125, 214)"><rect class="basic label-container" style="" x="-96.265625" y="-27" width="192.53125" height="54"></rect><g class="label" style="" transform="translate(-66.265625, -12)"><rect></rect><foreignobject width="132.53125" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
DecisionEngine.hs
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-DP-2" transform="translate(371.37109375, 343)"><rect class="basic label-container" style="" x="-94.7734375" y="-27" width="189.546875" height="54"></rect><g class="label" style="" transform="translate(-64.7734375, -12)"><rect></rect><foreignobject width="129.546875" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
DuplicatePolicy.hs
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-VAL-3" transform="translate(591.36328125, 343)"><rect class="basic label-container" style="" x="-75.21875" y="-27" width="150.4375" height="54"></rect><g class="label" style="" transform="translate(-45.21875, -12)"><rect></rect><foreignobject width="90.4375" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Validation.hs
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-SUM-4" transform="translate(790.88671875, 343)"><rect class="basic label-container" style="" x="-74.3046875" y="-27" width="148.609375" height="54"></rect><g class="label" style="" transform="translate(-44.3046875, -12)"><rect></rect><foreignobject width="88.609375" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Summary.hs
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-DEC-5" transform="translate(591.36328125, 497)"><rect class="basic label-container" style="" x="-90.46875" y="-27" width="180.9375" height="54"></rect><g class="label" style="" transform="translate(-60.46875, -12)"><rect></rect><foreignobject width="120.9375" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Domain.Decision
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-DUP-6" transform="translate(208.87109375, 601)"><rect class="basic label-container" style="" x="-93.140625" y="-27" width="186.28125" height="54"></rect><g class="label" style="" transform="translate(-63.140625, -12)"><rect></rect><foreignobject width="126.28125" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Domain.Duplicate
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-ROW-7" transform="translate(945.46875, 601)"><rect class="basic label-container" style="" x="-75.796875" y="-27" width="151.59375" height="54"></rect><g class="label" style="" transform="translate(-45.796875, -12)"><rect></rect><foreignobject width="91.59375" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Domain.Row
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-VEH-8" transform="translate(1156.8515625, 601)"><rect class="basic label-container" style="" x="-85.5859375" y="-27" width="171.171875" height="54"></rect><g class="label" style="" transform="translate(-55.5859375, -12)"><rect></rect><foreignobject width="111.171875" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Domain.Vehicle
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-PRIM-9" transform="translate(549.7421875, 705)"><rect class="basic label-container" style="" x="-90.4609375" y="-27" width="180.921875" height="54"></rect><g class="label" style="" transform="translate(-60.4609375, -12)"><rect></rect><foreignobject width="120.921875" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Domain.Primitive
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-AUD-10" transform="translate(1025.078125, 343)"><rect class="basic label-container" style="" x="-58.90625" y="-27" width="117.8125" height="54"></rect><g class="label" style="" transform="translate(-28.90625, -12)"><rect></rect><foreignobject width="57.8125" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Audit.hs
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-REP-11" transform="translate(1198.671875, 343)"><rect class="basic label-container" style="" x="-64.6875" y="-27" width="129.375" height="54"></rect><g class="label" style="" transform="translate(-34.6875, -12)"><rect></rect><foreignobject width="69.375" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Report.hs
</p>
</span>
</div>
</foreignobject></g></g></g></g></g>
</svg>
</div>

Two observations worth filing away:

1. **`Domain.Primitive` is the leaf.** The newtypes
   (`RowNumber`, `TransactionId`, `MoneyAmount`) and closed enums
   (`UploadMode`, `FatalError`) sit at the bottom. Everything depends
   on them. Change one, the whole library rebuilds — which is exactly
   what you want, because they're foundational.

2. **`Api` is the only module that touches strings or the outside
   world.** Everything else trades in fully-typed values. If you
   ever found `import Data.Char` in a `Domain.*` file, something
   would be wrong. The module structure is the boundary.

The file paths match the module names *exactly* —
`FuelUpload.Domain.Decision` lives at
`src/FuelUpload/Domain/Decision.hs`. The compiler enforces it. There's
no "search by namespace" indirection; the module name *is* the path.

## Reading `Domain/Decision.hs`

The whole file is about 115 lines. The module header lists the
exports — `RowDecision (..)` means "the type *and* its
constructors"; omit the dots and constructors stay private. Then
five `data` declarations for the supporting types
(`FuelTransaction`, `ValidationError`, `ValidationWarning`,
`QuarantineReason`, `RejectionReason`), each carrying the specific
data its case needs. `QuantityExceedsMaximum quantity limit` records
both the offending quantity *and* the configured limit — so the
rejection downstream can quote both without parsing them out of a
message string.

The centerpiece:

In [ ]:
data RowDecision
  = Accepted FuelTransaction
  | AcceptedWithWarnings FuelTransaction (NonEmpty ValidationWarning)
  | Quarantined FuelTransaction (NonEmpty QuarantineReason)
  | SkippedDuplicate SkippedDuplicate
  | Rejected RejectedRow
  | Fatal FatalError
  deriving stock (Eq, Show)

Six constructors, each carrying exactly what its case needs and no
more:

- `Accepted` carries the finished transaction. No warnings, no errors.
- `AcceptedWithWarnings` carries a transaction *and* a `NonEmpty
  ValidationWarning` — the compiler refuses to build it with an empty
  warning list.
- `Quarantined` is the same shape with `NonEmpty QuarantineReason`. A
  zero-reason row literally cannot be `Quarantined`.
- `SkippedDuplicate`, `Rejected`, and `Fatal` each carry their typed
  payload.

There is no "status string" anywhere — the constructor name *is* the
status. There are no nullable accessors. There are no empty-list
"optional" warnings.

Compare to the normal-C# version from chapter 2:

In [ ]:
public class RowDecision
{
    public int RowNumber;
    public string Status;
    public FuelTransaction Transaction;       // null when Status is Rejected / Fatal / Skipped
    public Vehicle Vehicle;                   // null when vehicle lookup failed
    public List<string> Errors = new List<string>();
    public List<string> Warnings = new List<string>();
    public List<string> QuarantineReasons = new List<string>();
    public string SkipReason;
    public string FatalMessage;
}

Nine fields, four of which are nullable, three of which are mutable
lists. The shape itself doesn't tell you which combinations are
valid. The Haskell version doesn't have invalid combinations.

## Reading a slice of `DecisionEngine.hs`

Now the function that produces a `RowDecision`. Two key passages.
First, the top-level signature and the fatal-error gate:

In [ ]:
classifyRow :: ValidationConfig -> UploadMode -> RowContext -> RowDecision
classifyRow config mode context =
  case (contextVehicleLookup context, contextDuplicateCheck context) of
    (VehicleLookupFatal fatalError, _) ->
      Fatal fatalError
    (_, DuplicateCheckFatal fatalError) ->
      Fatal fatalError
    _ ->
      classifyValidated

A 2-tuple pattern match handles the fatal cases first. If either the
vehicle lookup or the duplicate check came back fatal, the row is
`Fatal`. Otherwise — the wildcard branch — we drop into the local
`classifyValidated` helper.

The recovery matrix lives in `DuplicatePolicy.hs` as one function
defined by multiple equations, with **guards** (`|`) refining each
one. A guard is a boolean condition; the first `True` one wins, and
`otherwise` is a conventional name for `True`. Here are the
`Retry` and `AggressiveRecovery` equations:

In [ ]:
duplicateDecision Retry (DuplicateOf previousAttempt)
  | previousFinalizationState previousAttempt == FailedRetryable =
      UploadDuplicate
  | otherwise =
      SkipDuplicate (skipReasonForPreviousAttempt previousAttempt)
duplicateDecision AggressiveRecovery (DuplicateOf previousAttempt)
  | previousCanonicalizationState previousAttempt == FailedBeforeCanonicalization =
      UploadDuplicate
  | previousCanonicalizationState previousAttempt == CanonicalizedWithoutTransactionKey
      && previousFinalizationState previousAttempt /= Finalized =
      UploadDuplicate
  | otherwise =
      SkipDuplicate (skipReasonForPreviousAttempt previousAttempt)

Same matrix as the normal-C# `ShouldAcceptDuplicate` from chapter 2
(out-bool, string compares, nine branches over a flat surface). Same
logic. The Haskell version *is* the matrix, written as patterns over
the algebraic types. Add `ManualOverride` to `UploadMode` and the
compiler tells you exactly where the new equation has to go — that's
chapter 8's territory.

Finally, the warning/quarantine post-step, with the `NonEmpty`
construction inline:

In [ ]:
acceptedOrQuarantined vehicle =
  let transaction = toTransaction row vehicle
   in case quarantineReasons config row of
        firstReason : remainingReasons ->
          Quarantined transaction (firstReason :| remainingReasons)
        [] ->
          case validationWarnings config row of
            firstWarning : remainingWarnings ->
              AcceptedWithWarnings transaction (firstWarning :| remainingWarnings)
            [] ->
              Accepted transaction

`firstReason : remainingReasons` is **list pattern syntax** — the
`:` operator (read "cons") destructures a list into its head and
tail. Matching the regular list `[QuarantineReason]` as
`firstReason : remainingReasons` proves to the compiler that at
least one element exists, and we use `:|` (the `NonEmpty`
constructor) to repackage that proof. If we tried to write
`Quarantined transaction (someEmptyList)`, the program would not
compile.

That's the whole trick. The empty-list case is a structurally
separate branch that produces a different `RowDecision` constructor
(`Accepted` or `AcceptedWithWarnings`). You **cannot** end up at the
`Quarantined` branch holding an empty list. The types don't allow it.

## What's next

You can now read most of the Haskell files in this repo. The
exceptions are the applicative-style combinators (`<$>`, `<*>`) in
`Api.hs`, which we'll skim past, and the QuickCheck generators in
`Properties.hs`, which we'll meet next chapter.

Chapter 8 is the one this primer was setting up. Now we walk the
seven-footgun catalogue across Haskell's type system and ask the
question that's only askable up on the cliff face: **is there a value
of any type in the program that could even represent the bug?**

[next →](08-haskell-safety.ipynb)